# Решение тестового задания

## Загрузка данных

In [1]:
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from metric import precision_at_recall

In [2]:
train = pd.read_csv('data/train.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
test = pd.read_csv('data/test.csv', parse_dates=['cookie_created_at', 'window_start_ts', 'window_end_ts'])
events = pd.read_csv('data/events.csv.gz', parse_dates=['event_ts'])

print(train.shape, test.shape, events.shape)
print('доля ботов в train:', train.target.mean().round(4))
train.head()

(11091, 5) (4909, 4) (328905, 14)
доля ботов в train: 0.0811


,cookie_id,cookie_created_at,window_start_ts,window_end_ts,target
0,ck_54a059eb7d3ea68b,2025-11-21 09:30:41,2026-04-06,2026-04-07,0
1,ck_7e4de46eeab82974,2025-09-23 10:10:24,2026-04-06,2026-04-07,0
2,ck_9320229ef6304522,2026-03-04 00:08:02,2026-04-06,2026-04-07,0
3,ck_30ccd25bc1714ed9,2026-04-05 10:52:40,2026-04-06,2026-04-07,0
4,ck_a77c5f05948cdeef,2026-01-01 03:54:35,2026-04-06,2026-04-07,0


In [3]:
events.head()

,cookie_id,event_ts,eid,event_name,platform,user_agent,item_id,item_category,item_location,seller_type,search_query,search_page,pointer_x,pointer_y
0,ck_5efbea1befdefe1b,2026-04-26 09:11:24,200,item_view,desktop,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,1027450.0,elektronika,kaliningrad,pro,NaN,NaN,NaN,NaN
1,ck_c4ca1434f3778f1d,2026-04-20 14:04:31,100,search_results_view,web,Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:1...,NaN,telefony,habarovsk,NaN,iphone 13 128,4.0,NaN,NaN
2,ck_d274382b19488771,2026-04-13 12:02:56,200,item_view,WEB,Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,1048743.0,kvartiry_prodazha,kirov,private,NaN,NaN,675.0,276.0
3,ck_fbbed2ff14944ce9,2026-04-08 06:12:14,100,search_results_view,web,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,bytovaya_tehnika,novosibirsk,NaN,пылесос dyson,2.0,NaN,NaN
4,ck_56cc15c7c634cb9f,2026-04-12 17:16:05,100,search_results_view,WEB,Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7...,NaN,odezhda,sankt-peterburg,NaN,костюм мужской,2.0,NaN,NaN


Признаки считаем только по событиям внутри окна: `window_start_ts <= event_ts < window_end_ts`.
Это требование из условия, а не рекомендация.

In [4]:
def events_in_window(events, meta):
    ev = events.merge(meta[['cookie_id', 'window_start_ts', 'window_end_ts']], on='cookie_id')
    return ev[(ev.event_ts >= ev.window_start_ts) & (ev.event_ts < ev.window_end_ts)]

ev_tr = events_in_window(events, train)
ev_te = events_in_window(events, test)
print(len(ev_tr), len(ev_te))

198436 89690


## Анализ входных данных

In [5]:
print(events.event_name.value_counts(), '\n')
print(events.platform.value_counts(), '\n')
print('пропуски по колонкам:')
print(events.isna().mean().round(3))

event_name
item_view               120817
search_results_view     100402
photo_swipe              36517
favorite_add             19049
seller_page_view         17403
contact_phone_show       11316
captcha_shown             7928
login                     6267
contact_chat_open         6125
contact_message_sent      3081
Name: count, dtype: int64 

platform
desktop    46866
WEB        46476
Web        46365
web        45951
ANDROID    42462
Android    42254
android    42159
iphone      4103
IOS         4100
iOS         4091
ios         4078
Name: count, dtype: int64 

пропуски по колонкам:
cookie_id        0.000
event_ts         0.000
eid              0.000
event_name       0.000
platform         0.000
user_agent       0.000
item_id          0.348
item_category    0.100
item_location    0.072
seller_type      0.407
search_query     0.695
search_page      0.695
pointer_x        0.670
pointer_y        0.670
dtype: float64


Дальше будем работать по признакам, посмотрим на них, поймем, что в них присутствует, вычленим важную информацию для создания фич

### Начнем с названий действий (event_name).

Как уже узнали в ячейке выше, у нас может быть несколько различных значений:

1. item_view - просмотр объявления

2. search_results_view - просмотр выдачи поиска

3. photo_swipe - листание фотографий

4. favorite_add - добавление в избранное

5. seller_page_view - переход на страницу продавца

6. contact_phone_show - показ телефона

7. captcha_shown - показ капчи

8. login - вход в аккаунт

9. contact_chat_open - открытие чата

10. contact_message_sent - отправка сообщения

Стоит посмотреть на распределение событий внутри каждого класса, так сможем наглядно посмотреть, какой трафик делают в основном боты, а какой - люди.

Приклеим таргет к ev_tr

In [6]:
ev_tr_target = ev_tr.merge(train[['cookie_id', 'target']], on='cookie_id')

counts = ev_tr_target.groupby(['target', 'event_name']).size().unstack(fill_value=0)

shares = counts.div(counts.sum(axis=1), axis=0).round(4)

print("Абсолютные количества (строки — target, столбцы — event_name):")
print(counts)
print()
print("Доли внутри класса (строки — target, столбцы — event_name):")
print(shares)

Абсолютные количества (строки — target, столбцы — event_name):
event_name  contact_chat_open  contact_message_sent  contact_phone_show  \
target                                                                    
0                        3404                  1769                6361   
1                         376                   210                 685   

event_name  favorite_add  item_view  login  photo_swipe  search_results_view  \
target                                                                         
0                  11317      62770   3693        21091                52399   
1                    720      11568    207         1860                 9318   

event_name  seller_page_view  
target                        
0                       9125  
1                       1563  

Доли внутри класса (строки — target, столбцы — event_name):
event_name  contact_chat_open  contact_message_sent  contact_phone_show  \
target                                                 

Ого, а куда пропала капча. Проверим

In [7]:
print("События в трейне")
print(ev_tr.event_name.value_counts())
print()
print("События в тесте")
print(ev_te.event_name.value_counts())

События в трейне
event_name
item_view               74338
search_results_view     61717
photo_swipe             22951
favorite_add            12037
seller_page_view        10688
contact_phone_show       7046
login                    3900
contact_chat_open        3780
contact_message_sent     1979
Name: count, dtype: int64

События в тесте
event_name
item_view               33748
search_results_view     28081
photo_swipe             10133
favorite_add             5259
seller_page_view         4854
contact_phone_show       3242
contact_chat_open        1768
login                    1761
contact_message_sent      844
Name: count, dtype: int64


Видим, что капча вообще не влезает ни в одно из окон

Итак, посмотрим на доли распределения признаков между людьми и ботами. У ботов доли просмотра объявлений и выдачи поиска больше, чем у людей. Звучит логично, боты и направлены обычно на сбор информации и парсинг данных.

С другой стороны, у людей больше доля на остальных действиях, таких как свайп фото, добавление в любимое и тд.

Есть и признаки, где доли почти равны - например, просмотр страницы продавца. Такой признак сам по себе мало чего даст модели.

Напрашивается создание признаков долей по каждому типу события. Так и сделаем.

Также добавим бинарные признаки по исполнению действия в сессию у "человеческих" событий.

### Дальше платформа

Названия в разных регистрах - приведем все к нижнему. Посмотрим на распределение платформ между людьми и ботами

In [8]:
ev_tr_target['platform_norm'] = ev_tr_target['platform'].str.lower()

print("Уникальные platform после нормализации:")
print(ev_tr_target.platform_norm.value_counts())
print()

counts = ev_tr_target.groupby(['target', 'platform_norm']).size().unstack(fill_value=0)
shares = counts.div(counts.sum(axis=1), axis=0).round(4)

print("Абсолютные количества:")
print(counts)
print()
print("Доли внутри класса:")
print(shares)


platform_per_cookie = ev_tr_target.groupby('cookie_id').platform_norm.nunique()
print()
print("Сколько уникальных платформ у куки:")
print(platform_per_cookie.value_counts())

Уникальные platform после нормализации:
platform_norm
web        82152
android    77991
desktop    27855
ios         7836
iphone      2602
Name: count, dtype: int64

Абсолютные количества:
platform_norm  android  desktop   ios  iphone    web
target                                              
0                71784    22846  7422    2463  67414
1                 6207     5009   414     139  14738

Доли внутри класса:
platform_norm  android  desktop     ios  iphone     web
target                                                 
0               0.4175   0.1329  0.0432  0.0143  0.3921
1               0.2342   0.1890  0.0156  0.0052  0.5560

Сколько уникальных платформ у куки:
platform_norm
2    5728
1    5363
Name: count, dtype: int64


Во первых, объединим ios и iphone, это одно и то же. Дальше видим, чир у ботов везде больше доля на компах, будь то десктопная версия или версия в вебе (понятно, что можно в вебе и с телефона открыть, но это происходит сильно реже, чем в приложении). Видим, что ни одна куки не прыгает между многими платформами. Это не дает огромных различий между ботами и людьми, потому что почти все люди сидят и с компа, и с телефона.

Создадим признаки по долям использования платформ, сверху докинем бинарный признак "телефон ли", у людей телефона прям побольше.

In [9]:
n_platforms = ev_tr_target.groupby(['cookie_id', 'target']).platform_norm.nunique().reset_index()
n_platforms.columns = ['cookie_id', 'target', 'n_unique_platforms']

print("Распределение n_unique_platforms по классам:")
print(n_platforms.groupby(['target', 'n_unique_platforms']).size().unstack(fill_value=0))
print()

counts = n_platforms.groupby(['target', 'n_unique_platforms']).size().unstack(fill_value=0)
shares = counts.div(counts.sum(axis=1), axis=0).round(4)
print("Доли внутри класса:")
print(shares)

Распределение n_unique_platforms по классам:
n_unique_platforms     1     2
target                        
0                   5055  5137
1                    308   591

Доли внутри класса:
n_unique_platforms       1       2
target                            
0                   0.4960  0.5040
1                   0.3426  0.6574


Докинем признак количества уникальных платформ

### user_agent

Можем извлечь много полезных штук

In [10]:
print("Всего уникальных user_agent:", ev_tr_target.user_agent.nunique())
print("Всего событий:", len(ev_tr_target))
print()

print("Топ-20 user_agent по частоте:")
print(ev_tr_target.user_agent.value_counts().head(20))

Всего уникальных user_agent: 148
Всего событий: 198436

Топ-20 user_agent по частоте:
user_agent
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 YaBrowser/23.0.0.0 Safari/537.36     2330
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 YaBrowser/23.11.0.0 Safari/537.36    2254
Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36                  2252
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 YaBrowser/23.9.0.0 Safari/537.36     2182
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/123.0.0.0 YaBrowser/23.3.0.0 Safari/537.36     2181
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 YaBrowser/23.8.0.0 Safari/537.36     2169
Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (K

Ого, в топе признаков видим HeadlessChrome - очень сильный сигнал!! Глянем еще на распределение по классам.

In [11]:
ev_tr_target['is_headless'] = ev_tr_target.user_agent.str.contains('Headless', case=False)

print("HeadlessChrome по классам:")
print(ev_tr_target.groupby(['target', 'is_headless']).size().unstack(fill_value=0))
print()

HeadlessChrome по классам:
is_headless   False  True 
target                    
0            168433   3496
1             23000   3507



Доля ботов с headlessChrome реально огромная.

Глянем на распределения признаков по доле событий на каждой из операционных систем.

In [12]:
def parse_ua(ua):
    ua = ua.lower()
    if 'windows' in ua:
        os_name = 'windows'
    elif 'macintosh' in ua or 'mac os' in ua:
        os_name = 'macos'
    elif 'linux' in ua:
        os_name = 'linux'
    elif 'android' in ua:
        os_name = 'android'
    elif 'iphone' in ua or 'ipad' in ua:
        os_name = 'ios'
    else:
        os_name = 'other'

    if 'headless' in ua:
        browser = 'headless'
    elif 'yabrowser' in ua:
        browser = 'yabrowser'
    elif 'firefox' in ua:
        browser = 'firefox'
    elif 'chrome' in ua:
        browser = 'chrome'
    elif 'safari' in ua:
        browser = 'safari'
    else:
        browser = 'other'

    return os_name, browser

ev_tr_target[['os_name', 'browser']] = ev_tr_target.user_agent.apply(
    lambda ua: pd.Series(parse_ua(ua))
)

print("ОС по классам (доли):")
counts_os = ev_tr_target.groupby(['target', 'os_name']).size().unstack(fill_value=0)
print(counts_os.div(counts_os.sum(axis=1), axis=0).round(4))
print()

print("Браузеры по классам (доли):")
counts_br = ev_tr_target.groupby(['target', 'browser']).size().unstack(fill_value=0)
print(counts_br.div(counts_br.sum(axis=1), axis=0).round(4))

ОС по классам (доли):
os_name  android     ios   linux   macos   other  windows
target                                                   
0         0.1503  0.0386  0.2876  0.1460  0.0143   0.3632
1         0.1541  0.0118  0.2124  0.1254  0.0668   0.4296

Браузеры по классам (доли):
browser  chrome  firefox  headless   other  safari  yabrowser
target                                                       
0        0.5187   0.1238    0.0203  0.2031  0.0189     0.1151
1        0.3433   0.1571    0.1323  0.2327  0.0091     0.1255


Видно неплохое разделение и в ос, и в браузерах.

### Item_id

Здесь понятно, что он заполнен только для событий item_view. Для сессии можно собрать количество уникальных айдишек объявлений, отношение уникальных к общему количеству для просмотра разнообразия, повторные просмотры и среднее число просмотров на одно событие.

In [13]:
ev_items = ev_tr_target[ev_tr_target.item_id.notna()].copy()
print("Событий с item_id:", len(ev_items))
print("Из них у ботов:", len(ev_items[ev_items.target == 1]))
print("Из них у людей:", len(ev_items[ev_items.target == 0]))
print()

g = ev_items.groupby('cookie_id')
item_features = pd.DataFrame({
    'n_item_views': g.size(),
    'n_unique_items': g.item_id.nunique(),
})

item_features['unique_item_ratio'] = item_features.n_unique_items / item_features.n_item_views

item_features['avg_views_per_item'] = item_features.n_item_views / item_features.n_unique_items

item_features = item_features.reset_index().merge(
    train[['cookie_id', 'target']], on='cookie_id'
)

print("Средние значения по классам:")
print(item_features.groupby('target')[['n_item_views', 'n_unique_items', 'unique_item_ratio', 'avg_views_per_item']].mean().round(4))
print()
print("Медианы по классам:")
print(item_features.groupby('target')[['n_item_views', 'n_unique_items', 'unique_item_ratio', 'avg_views_per_item']].median().round(4))

Событий с item_id: 132819
Из них у ботов: 16982
Из них у людей: 115837

Средние значения по классам:
        n_item_views  n_unique_items  unique_item_ratio  avg_views_per_item
target                                                                     
0            11.5617          9.0712             0.8031              1.3058
1            18.8899         13.7798             0.7966              1.3097

Медианы по классам:
        n_item_views  n_unique_items  unique_item_ratio  avg_views_per_item
target                                                                     
0                8.0             6.0                0.8                1.25
1               13.0            10.0                0.8                1.25


Итак, у ботов количество просмотров объявлений больше. У них также больше уникальных значений.

Разницы в долях и в среднем числе просмотров на одно объявление практически нет.

### Категория

Заполнена для большинства событий, можно вычленить количество уникальных категорий

In [14]:
ev_cat = ev_tr_target[ev_tr_target.item_category.notna()].copy()
print("Событий с item_category:", len(ev_cat))
print("Из них у ботов:", len(ev_cat[ev_cat.target == 1]))
print("Из них у людей:", len(ev_cat[ev_cat.target == 0]))
print()

g = ev_cat.groupby('cookie_id')
cat_features = pd.DataFrame({
    'n_category_events': g.size(),
    'n_unique_categories': g.item_category.nunique(),
})

cat_features = cat_features.reset_index().merge(
    train[['cookie_id', 'target']], on='cookie_id'
)

print("Средние значения по классам:")
print(cat_features.groupby('target')[['n_category_events', 'n_unique_categories']].mean().round(4))
print()
print("Медианы по классам:")
print(cat_features.groupby('target')[['n_category_events', 'n_unique_categories']].median().round(4))
print()

print("Топ-15 категорий по частоте:")
print(ev_cat.item_category.value_counts().head(15))

Событий с item_category: 182802
Из них у ботов: 24679
Из них у людей: 158123

Средние значения по классам:
        n_category_events  n_unique_categories
target                                        
0                 15.5679               2.2891
1                 27.5128               1.7915

Медианы по классам:
        n_category_events  n_unique_categories
target                                        
0                    11.0                  2.0
1                    19.0                  2.0

Топ-15 категорий по частоте:
item_category
avtomobili           13158
mebel                12688
kvartiry_prodazha    12531
hobbi                12514
odezhda              12453
uslugi               12213
noutbuki             12178
elektronika          12162
telefony             12137
bytovaya_tehnika     12108
detskie_tovary       12063
rabota               11849
kvartiry_arenda      11848
zapchasti            11515
zhivotnye            11385
Name: count, dtype: int64


У ботиков количество категорий гораздо больше, тем не менее уникальных категорий больше у людей.

В качестве признака точно стоит взять количество категорий, ентропию категории (разброс у ботов, кажется, будет ниже). Докинем также и уникальные категории, так как у людей их больше, а еще добавим долю самой частой категории, ведь у ботов она должна быть выше за счет узкого разброса категорий.

### Локация

Идея в том, что боты смотрят объявления из большого пула городов, а люди только из близлежащих к ним

In [15]:
ev_loc = ev_tr_target[ev_tr_target.item_location.notna()].copy()

g = ev_loc.groupby('cookie_id')
loc_features = pd.DataFrame({
    'n_location_events': g.size(),
    'n_unique_locations': g.item_location.nunique(),
}).reset_index().merge(train[['cookie_id', 'target']], on='cookie_id')

print("Средние по классам:")
print(loc_features.groupby('target')[['n_location_events', 'n_unique_locations']].mean().round(4))
print()
print("Медианы по классам:")
print(loc_features.groupby('target')[['n_location_events', 'n_unique_locations']].median().round(4))

Средние по классам:
        n_location_events  n_unique_locations
target                                       
0                 16.0281              5.8905
1                 28.3560              9.7898

Медианы по классам:
        n_location_events  n_unique_locations
target                                       
0                    11.0                 5.0
1                    19.0                 8.0


Идея подтвердилась, докинем эту фичу

### Тип продавца

Эта фича имеет огромное количество пропусков, поэтому скорее всего просто просадит модель. Тем более следуя из простой логики огромной разницы между ботами и людьми в просмотре продавцов не будет

### Поисковый запрос и номер страницы

Заполнены только там, где тип ивента был поиск. Логично воткнуть количество событий с запросами, среднюю длину запроса, долю повторяющихся запросов.

Также посмотреть на ту страницу, которая в среднем попадалась и количество уникальных страниц. Логично, что боты могут бежать далеко, а вот люди листают не так много

### Координаты указателя

Идея в том, что у ботиков координаты могут быть пустыми или не меняться, в то время как люди тыкают в разные места

## Создание признаков

Начнем с базы, возьмем функцию из quickstart, на ней же и обучим тот самый бейзлайн

In [16]:
def basic_features(ev, meta):
    g = ev.groupby('cookie_id')
    f = pd.DataFrame({
        'n_events': g.size(),
        'item_nunique': g.item_id.nunique(),
    })
    f = meta[['cookie_id']].merge(f.reset_index(), on='cookie_id', how='left')
    return f.fillna(0)

Xtr = basic_features(ev_tr, train)
Xte = basic_features(ev_te, test)
ytr = train.target.values
Xtr.head()

,cookie_id,n_events,item_nunique
0,ck_54a059eb7d3ea68b,7,4
1,ck_7e4de46eeab82974,41,19
2,ck_9320229ef6304522,36,19
3,ck_30ccd25bc1714ed9,21,10
4,ck_a77c5f05948cdeef,29,16


In [17]:
def top_share(series):
  counts = series.value_counts()
  return counts.iloc[0] / counts.sum()


def entropy(series):
  counts = series.value_counts(normalize=True)
  return -(counts * np.log(counts + 1e-10)).sum()


def repeated_share(series):
  counts = series.value_counts()
  return 1 - (counts == 1).sum() / len(counts) if len(counts) > 0 else 0


def std_or_zero(series):
  return series.std() if series.notna().sum() > 1 else 0


def calc_intervals(group):
  if len(group) < 2:
    return pd.Series({
                'time_span': 0, 'mean_interval': 0, 'std_interval': 0,
                'median_interval': 0, 'min_interval': 0, 'max_interval': 0,
    })
  intervals = group.event_ts.diff().dt.total_seconds().dropna()
  return pd.Series({
            'time_span': (group.event_ts.max() - group.event_ts.min()).total_seconds(),
            'mean_interval': intervals.mean(),
            'std_interval': intervals.std() if len(intervals) > 1 else 0,
            'median_interval': intervals.median(),
            'min_interval': intervals.min(),
            'max_interval': intervals.max(),
    })


def event_name_features(ev, meta):

    event_types = [
        'item_view',
        'search_results_view',
        'photo_swipe',
        'favorite_add',
        'seller_page_view',
        'contact_phone_show',
        'login',
        'contact_chat_open',
        'contact_message_sent',
    ]

    human_events = [
        'login',
        'favorite_add',
        'contact_phone_show',
        'contact_chat_open',
        'contact_message_sent',
    ]

    g = ev.groupby('cookie_id')

    features = pd.DataFrame({
        'n_events': g.size(),
        'n_unique_event_types': g.event_name.nunique(),
    })

    counts = ev.groupby(['cookie_id', 'event_name']).size().unstack(fill_value=0)
    for et in event_types:
        if et not in counts.columns:
            counts[et] = 0
    counts = counts[event_types]
    shares = counts.div(features['n_events'], axis=0).fillna(0)
    shares.columns = [f'share_{c}' for c in shares.columns]

    flags = (counts[human_events] > 0).astype(int)
    flags.columns = [f'has_{c}' for c in flags.columns]

    result = pd.concat([features, shares, flags], axis=1).reset_index()
    result = meta[['cookie_id']].merge(result, on='cookie_id', how='left')
    result = result.fillna(0)

    return result


def platform_features(ev, meta):

    ev = ev.copy()
    ev['platform_norm'] = ev['platform'].str.lower()
    ev['platform_norm'] = ev['platform_norm'].replace({'iphone': 'ios'})

    platforms = ['web', 'android', 'desktop', 'ios']

    counts = ev.groupby(['cookie_id', 'platform_norm']).size().unstack(fill_value=0)
    for p in platforms:
        if p not in counts.columns:
            counts[p] = 0
    counts = counts[platforms]

    n_events = counts.sum(axis=1)
    shares = counts.div(n_events, axis=0).fillna(0)
    shares.columns = [f'share_platform_{p}' for p in shares.columns]

    n_unique = ev.groupby('cookie_id').platform_norm.nunique().rename('n_unique_platforms')

    share_mobile = (shares['share_platform_android'] + shares['share_platform_ios']).rename('share_mobile')

    result = pd.concat([shares, n_unique, share_mobile], axis=1).reset_index()
    result = meta[['cookie_id']].merge(result, on='cookie_id', how='left')
    result = result.fillna(0)

    return result


def ua_features(ev, meta):

    ev = ev.copy()

    ev[['os_name', 'browser']] = ev.user_agent.apply(lambda ua: pd.Series(parse_ua(ua)))

    ev['is_headless'] = ev.user_agent.str.contains('Headless', case=False).astype(int)

    os_counts = ev.groupby(['cookie_id', 'os_name']).size().unstack(fill_value=0)
    n_events = os_counts.sum(axis=1)
    os_shares = os_counts.div(n_events, axis=0).fillna(0)
    os_shares.columns = [f'share_os_{c}' for c in os_shares.columns]

    br_counts = ev.groupby(['cookie_id', 'browser']).size().unstack(fill_value=0)
    br_shares = br_counts.div(n_events, axis=0).fillna(0)
    br_shares.columns = [f'share_browser_{c}' for c in br_shares.columns]

    has_headless = (ev.groupby('cookie_id').is_headless.sum() > 0).astype(int).rename('has_headless')

    result = pd.concat([os_shares, br_shares, has_headless], axis=1).reset_index()
    result = meta[['cookie_id']].merge(result, on='cookie_id', how='left')
    result = result.fillna(0)

    return result


def item_features(ev, meta):

    ev_items = ev[ev.item_id.notna()].copy()

    g = ev_items.groupby('cookie_id')
    features = pd.DataFrame({
        'n_item_views': g.size(),
        'n_unique_items': g.item_id.nunique(),
    }).reset_index()

    result = meta[['cookie_id']].merge(features, on='cookie_id', how='left')
    result = result.fillna(0)

    return result


def category_features(ev, meta):

    ev_cat = ev[ev.item_category.notna()].copy()

    g = ev_cat.groupby('cookie_id')
    features = pd.DataFrame({
        'n_category_events': g.size(),
        'n_unique_categories': g.item_category.nunique(),
    })

    top_shares = g.item_category.apply(top_share).rename('share_most_common_category')
    entropies = g.item_category.apply(entropy).rename('category_entropy')

    result = pd.concat([features, top_shares, entropies], axis=1).reset_index()
    result = meta[['cookie_id']].merge(result, on='cookie_id', how='left')
    result = result.fillna(0)

    return result


def location_features(ev, meta):

    ev_loc = ev[ev.item_location.notna()].copy()

    g = ev_loc.groupby('cookie_id')
    features = pd.DataFrame({
        'n_location_events': g.size(),
        'n_unique_locations': g.item_location.nunique(),
    })

    entropies = g.item_location.apply(entropy).rename('location_entropy')
    top_shares = g.item_location.apply(top_share).rename('share_most_common_location')

    result = pd.concat([features, entropies, top_shares], axis=1).reset_index()
    result = meta[['cookie_id']].merge(result, on='cookie_id', how='left')
    result = result.fillna(0)

    return result


def search_features(ev, meta):

    ev_s = ev[ev.search_query.notna()].copy()

    if len(ev_s) == 0:
        result = meta[['cookie_id']].copy()
        for col in ['n_search_events', 'n_unique_queries', 'avg_query_length',
                    'share_repeated_queries', 'max_search_page', 'avg_search_page',
                    'n_unique_search_pages']:
            result[col] = 0
        return result

    g = ev_s.groupby('cookie_id')

    features = pd.DataFrame({
        'n_search_events': g.size(),
        'n_unique_queries': g.search_query.nunique(),
        'avg_query_length': g.search_query.apply(lambda x: x.str.len().mean()),
        'max_search_page': g.search_page.max(),
        'avg_search_page': g.search_page.mean(),
        'n_unique_search_pages': g.search_page.nunique(),
    })

    features['share_repeated_queries'] = g.search_query.apply(repeated_share)

    result = features.reset_index()
    result = meta[['cookie_id']].merge(result, on='cookie_id', how='left')
    result = result.fillna(0)

    return result


def pointer_features(ev, meta):

    ev_p = ev.copy()
    ev_p['has_pointer'] = ev_p.pointer_x.notna().astype(int)

    g = ev_p.groupby('cookie_id')

    features = pd.DataFrame({
        'n_pointer_events': g.has_pointer.sum(),
        'share_pointer_events': g.has_pointer.mean(),
        'n_unique_pointer_x': g.pointer_x.nunique(),
        'n_unique_pointer_y': g.pointer_y.nunique(),
    })

    features['std_pointer_x'] = g.pointer_x.apply(std_or_zero)
    features['std_pointer_y'] = g.pointer_y.apply(std_or_zero)

    result = features.reset_index()
    result = meta[['cookie_id']].merge(result, on='cookie_id', how='left')
    result = result.fillna(0)
    return result


def temporal_features(ev, meta):

    ev_t = ev.sort_values(['cookie_id', 'event_ts']).copy()

    g = ev_t.groupby('cookie_id')
    interval_features = g.apply(calc_intervals)

    n_events = g.size()
    interval_features['events_per_minute'] = n_events / (interval_features.time_span / 60 + 1e-6)

    ev_t['hour'] = ev_t.event_ts.dt.hour
    ev_t['is_night'] = ev_t.hour.between(2, 5).astype(int)
    night_features = pd.DataFrame({
        'share_night_events': ev_t.groupby('cookie_id').is_night.mean(),
        'n_unique_hours': ev_t.groupby('cookie_id').hour.nunique(),
    })

    result = pd.concat([interval_features, night_features], axis=1).reset_index()
    result = meta[['cookie_id']].merge(result, on='cookie_id', how='left')
    result = result.fillna(0)
    return result

In [18]:
features = {
    'event':    event_name_features,
    'platform': platform_features,
    'ua':       ua_features,
    'item':     item_features,
    'cat':      category_features,
    'loc':      location_features,
    'search':   search_features,
    'pointer':  pointer_features,
    'temporal': temporal_features,
}

Xtr_combined = train[['cookie_id']].copy()
Xte_combined = test[['cookie_id']].copy()

for name, func in features.items():
    Xtr_combined = Xtr_combined.merge(func(ev_tr, train), on='cookie_id')
    Xte_combined = Xte_combined.merge(func(ev_te, test), on='cookie_id')

print("Xtr_combined:", Xtr_combined.shape)
print("Xte_combined:", Xte_combined.shape)

/tmp/ipykernel_2024/1984016128.py:255: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  interval_features = g.apply(calc_intervals)


Xtr_combined: (11091, 68)
Xte_combined: (4909, 68)


/tmp/ipykernel_2024/1984016128.py:255: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  interval_features = g.apply(calc_intervals)


## Модели

### Бейзлайн из quickstart

In [19]:
is_valid = train.window_start_ts.ge('2026-04-17').values
cols = ['n_events', 'item_nunique']

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model.fit(Xtr.loc[~is_valid, cols], ytr[~is_valid])
p_va = model.predict_proba(Xtr.loc[is_valid, cols])[:, 1]

print('P@R0.7 на валидации:', round(precision_at_recall(ytr[is_valid], p_va), 4))
print('доля ботов (это уровень константы):', round(ytr[is_valid].mean(), 4))

P@R0.7 на валидации: 0.1025
доля ботов (это уровень константы): 0.082


### Скомбинированная моделька с базовыми параметрами

In [20]:
print("Размер Xtr_combined:", Xtr_combined.shape)

is_valid = train.window_start_ts.ge('2026-04-17').values
cols = [c for c in Xtr_combined.columns if c != 'cookie_id']

model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, random_state=0)
model.fit(Xtr_combined.loc[~is_valid, cols], train.target.values[~is_valid])
p_va = model.predict_proba(Xtr_combined.loc[is_valid, cols])[:, 1]

print('P@R0.7:', round(precision_at_recall(train.target.values[is_valid], p_va), 4))

Размер Xtr_combined: (11091, 68)
P@R0.7: 0.7


## Сабмит

In [22]:
cols_final = [c for c in Xtr_combined.columns if c != 'cookie_id']

final_model = RandomForestClassifier(
    n_estimators=500,
    min_samples_leaf=3,
    random_state=0,
    n_jobs=-1
)
final_model.fit(Xtr_combined[cols_final], train.target.values)

scores = final_model.predict_proba(Xte_combined[cols_final])[:, 1]

sub = pd.DataFrame({
    'cookie_id': Xte_combined.cookie_id,
    'score': scores,
})

assert len(sub) == len(test), f"Длина не совпадает: {len(sub)} vs {len(test)}"
assert sub.score.between(0, 1).all(), "Есть скоры вне [0, 1]"
assert sub.cookie_id.nunique() == len(sub), "Есть дубликаты cookie_id"

sub.to_csv('submission.csv', index=False)
print(f"Строк: {len(sub)}")
print(f"Скор: min={sub.score.min():.4f}, max={sub.score.max():.4f}, mean={sub.score.mean():.4f}")
print()
print(sub.head(10))

Строк: 4909
Скор: min=0.0000, max=0.9990, mean=0.0893

             cookie_id     score
0  ck_315fb710a0e371e7  0.010805
1  ck_a76ee3b3e3e522fd  0.246536
2  ck_94c9a4d382689e82  0.005178
3  ck_8eaf9509ad9462a0  0.014297
4  ck_9a88a5a989cb5bc6  0.008929
5  ck_15239ef0689c1a35  0.045777
6  ck_8de69cd219d3b87f  0.149079
7  ck_f92cd37022b0ce21  0.021517
8  ck_1a3ff7620580a514  0.007046
9  ck_397c03aa33a57310  0.271477


Пока отправляю такой, попробую успеть еще докинуть перебор гиперпараметров